In [ ]:
import json
from pathlib import Path

import pandas as pd
from statsmodels.tsa.stattools import kpss

from google.colab import drive
drive.mount('/content/drive')

config_path = Path('/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json')
csv_dir = Path("/content/drive/MyDrive/data/FINAL SELECTED DATA SET/clean data")

with config_path.open(encoding="utf-8") as f:
    config = json.load(f)

results = []

for dataset in config["datasets"]:
    csv_path = csv_dir / Path(dataset["file_path"]).name

    if not csv_path.exists():
        print(f"CSV پیدا نشد: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    time_col = dataset["time_column"]

    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.sort_values(time_col)

    for target in dataset["targets"]:
        column = target["column"]

        if column not in df.columns:
            print(f"ستون پیدا نشد: {dataset['dataset_name']} / {column}")
            continue

        series = pd.to_numeric(df[column], errors="coerce").dropna()

        if len(series) < 3:
            print(f"دادهٔ کافی نیست: {dataset['dataset_name']} / {column}")
            continue

        try:
            # Call kpss and unpack the returned tuple directly instead of using result_object
            statistic, pvalue, lags, critical_values = kpss(
                series,
                regression="c",
                nlags="auto"
            )

            results.append({
                "dataset": dataset["dataset_name"],
                "column": column,
                "n": len(series),
                "statistic": statistic,
                "pvalue": pvalue,
                "lags": lags,
                "critical_values": critical_values,
            })

        except Exception as e:
            print(f"خطا برای {dataset['dataset_name']} / {column}: {e}")

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

results_df.to_csv("kpss_results_26_series.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset         column   n  statistic   pvalue  lags                                         critical_values
   WB90             wt 531   2.110197 0.010000    14 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90            toc 531   1.595750 0.010000     9 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90    temperature 531   0.418042 0.069379    14 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90             ph 531   3.815293 0.010000     9 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90       humidity 531   0.149255 0.100000    14 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90             do 531   2.719082 0.010000    15 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
   WB90            cod 531   1.572413 0.010000     1 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}

/tmp/ipython-input-4-2286309052.py:48: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, pvalue, lags, critical_values = kpss(
/tmp/ipython-input-4-2286309052.py:48: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, pvalue, lags, critical_values = kpss(
/tmp/ipython-input-4-2286309052.py:48: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, pvalue, lags, critical_values = kpss(
/tmp/ipython-input-4-2286309052.py:48: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  statistic, pvalue, la

In [5]:
import json
from pathlib import Path

import pandas as pd
from statsmodels.tsa.stattools import adfuller


from google.colab import drive
drive.mount('/content/drive')

config_path = Path('/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json')
csv_dir = Path("/content/drive/MyDrive/data/FINAL SELECTED DATA SET/clean data")


with config_path.open(encoding="utf-8") as f:
    config = json.load(f)

results = []

for dataset in config["datasets"]:
    csv_path = csv_dir / Path(dataset["file_path"]).name

    if not csv_path.exists():
        print(f"CSV پیدا نشد: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    time_col = dataset["time_column"]

    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.sort_values(time_col)

    for target in dataset["targets"]:
        column = target["column"]

        if column not in df.columns:
            print(f"ستون پیدا نشد: {dataset['dataset_name']} / {column}")
            continue

        series = pd.to_numeric(df[column], errors="coerce").dropna()

        if len(series) < 10:
            print(f"دادهٔ کافی نیست: {dataset['dataset_name']} / {column}")
            continue

        try:
            statistic, pvalue, usedlag, nobs, critical_values, icbest = adfuller(
                series,
                regression="c",
                autolag="AIC",
            )

            results.append({
                "dataset": dataset["dataset_name"],
                "column": column,
                "n": len(series),
                "statistic": statistic,
                "pvalue": pvalue,
                "lags": usedlag,
                "nobs": nobs,
                "critical_values": critical_values,
                "icbest": icbest,
            })

        except Exception as e:
            print(f"خطا برای {dataset['dataset_name']} / {column}: {e}")

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv("adf_results_26_series.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset         column   n  statistic       pvalue  lags  nobs                                                                    critical_values       icbest
   WB90             wt 531  -1.918217 3.235404e-01    19   511  {'1%': -3.4432119442564324, '5%': -2.8672126791646955, '10%': -2.569791324979607}  -224.365127
   WB90            toc 531  -7.991023 2.474349e-12     5   525 {'1%': -3.4428672600488066, '5%': -2.8670609714890403, '10%': -2.5697104770975057} -1114.537099
   WB90    temperature 531  -2.091348 2.480076e-01    19   511  {'1%': -3.4432119442564324, '5%': -2.8672126791646955, '10%': -2.569791324979607}   838.111522
   WB90             ph 531  -3.013173 3.368237e-02    11   519  {'1%': -3.4430126933746767, '5%': -2.8671249839002764, '10%': -2.569744590233924} -1301.860100
   WB90       humidity 531  -1.382651 5.905648e-01    19   511  {'1%': -3.44

In [7]:
!pip install antropy

import json
from pathlib import Path

import antropy as ant
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

config_path = Path('/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json')
csv_dir = Path("/content/drive/MyDrive/data/FINAL SELECTED DATA SET/clean data")

with config_path.open(encoding="utf-8") as f:
    config = json.load(f)

results = []

for dataset in config["datasets"]:
    csv_path = csv_dir / Path(dataset["file_path"]).name

    if not csv_path.exists():
        print(f"CSV پیدا نشد: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    time_col = dataset["time_column"]

    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.sort_values(time_col)

    for target in dataset["targets"]:
        column = target["column"]

        if column not in df.columns:
            print(f"ستون پیدا نشد: {dataset['dataset_name']} / {column}")
            continue

        x = pd.to_numeric(df[column], errors="coerce").dropna().to_numpy()

        if len(x) < 10:
            print(f"دادهٔ کافی نیست: {dataset['dataset_name']} / {column}")
            continue

        # آنتروپی طیفی نرمال‌شده در بازهٔ 0 تا 1
        spectral_entropy = ant.spectral_entropy(
            x,
            sf=1,
            method="welch",
            normalize=True,
        )

        # Omega در بازهٔ 0 تا 1
        omega = 1 - spectral_entropy

        results.append({
            "dataset": dataset["dataset_name"],
            "column": column,
            "n": len(x),
            "spectral_entropy": spectral_entropy,
            "Omega_0_1": omega,
            "Omega_percent": 100 * omega,
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv("/content/forecastability_26_series.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset         column   n  spectral_entropy  Omega_0_1  Omega_percent
   WB90             wt 531          0.561979   0.438021      43.802088
   WB90            toc 531          0.905890   0.094110       9.411008
   WB90    temperature 531          0.440729   0.559271      55.927071
   WB90             ph 531          0.852835   0.147165      14.716505
   WB90       humidity 531          0.501356   0.498644      49.864430
   WB90             do 531          0.772837   0.227163      22.716292
   WB90            cod 531          0.877903   0.122097      12.209701
   WB90             cl 531          0.732172   0.267828      26.782763
   WB90            bod 531          0.881593   0.118407      11.840667
   WB90 air_wind_speed 531          0.535551   0.464449      46.444860
   WB90       air_temp 531          0.459027   0.540973      54.097282
   WB90   air_press

/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 184, using nperseg = 184
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
